<a target="_blank" href="https://colab.research.google.com/github/instadeepai/jumanji/blob/main/examples/training.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

In [21]:
%%capture
!pip install --quiet -U "jumanji[train] @ git+https://github.com/drinwater/jumanji.git@main"

In [ ]:
# @title Download Configs (run me) { display-mode: "form" }

import os
import requests


def download_file(url: str, file_path: str) -> None:
    # Send an HTTP GET request to the URL
    response = requests.get(url)
    # Check if the request was successful (status code 200)
    if response.status_code == 200:
        with open(file_path, "wb") as f:
            f.write(response.content)
    else:
        print("Failed to download the file.")


os.makedirs("configs", exist_ok=True)
env_url = f"https://raw.githubusercontent.com/drinwater/jumanji/main/jumanji/training/configs/env/{env}.yaml"
os.makedirs("configs/env", exist_ok=True)
download_file(env_url, f"configs/env/{env}.yaml")

In [8]:
import jax

import jumanji
from jumanji.wrappers import AutoResetWrapper

env = jumanji.make("Connector-v3")  # Create a Snake environment
env = AutoResetWrapper(env)     # Automatically reset the environment when an episode terminates

batch_size = 100
rollout_length = 10
num_actions = env.action_spec.num_values

random_key = jax.random.PRNGKey(0)
key1, key2 = jax.random.split(random_key)

def step_fn(state, key):
  action = jax.random.randint(key=key, minval=0, maxval=num_actions, shape=(10,))
  new_state, timestep = env.step(state, action)
  return new_state, timestep

def run_n_steps(state, key, n):
  random_keys = jax.random.split(key, n)
  rollout = jax.lax.scan(step_fn, state, random_keys)
  return rollout

# Instantiate a batch of environment states
keys = jax.random.split(key1, batch_size)
state, timestep = jax.vmap(env.reset)(keys)

# Collect a batch of rollouts
keys = jax.random.split(key2, batch_size)
rollout = jax.vmap(run_n_steps, in_axes=(0, 0, None))(state, keys, rollout_length)

# Shape and type of given rollout:
# TimeStep(step_type=(7, 5), reward=(7, 5), discount=(7, 5), observation=(7, 5, 6, 6, 5), extras=None)

In [17]:
from jumanji.environments.routing.connector import State

observations = rollout["observation"]
grids = observations.grid.reshape(1, 1000, 10, 10).at[0, :, :, :]
steps = observations.step_count.reshape(1, 1000, 10, 10).at[0, :, :, :]
actions = observations.action_mask.reshape(1, 1000, 10, 10).at[0, :, :, :]

for i in range(1000):
  state = State(
    grid=grids.at[i, :, :],
    step_count=steps.at[i, :, :],
    action_mask=actions.at[i, :, :]
)

(100, 10, 10, 10)

In [23]:
import jax

import jumanji
from jumanji.wrappers import AutoResetWrapper

env = jumanji.make("Connector-v3")  # Create a Snake environment
env = AutoResetWrapper(env)     # Automatically reset the environment when an episode terminates

In [ ]:
from IPython.core.display import clear_output
learning_rate = 0.01
n_episodes = 20

import tqdm
states = []
for episode in tqdm.tqdm(range(n_episodes)):
  done = False
  key = jax.random.PRNGKey(0)
  state, timestep = env.reset(key)

  while not timestep.last():
    action = jax.random.randint(key=key, minval=0, maxval=5, shape=(10,))
    state, timestep = env.step(state, action)
    states.append(state)
    # env.render(state)

  0%|          | 0/20 [00:00<?, ?it/s]

In [29]:
env.animate(states)